
<a target="_blank" href="https://colab.research.google.com/github/hysebsch/tirex-2/blob/main/examples/pro_features.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook In Google Colab"/>
</a>

# TiRex-2 Pro Features

This notebook demonstrates the **Pro capabilities** built on top of the TiRex-2 forecasting backbone:

1. **Streaming / incremental forecasting** with `IncrementalForecaster`
2. **Time-series regression** with `TimeSeriesRegressor`
3. **Multivariate anomaly detection** with `TimeSeriesAnomalyDetector`

All examples run on CPU with the public `NX-AI/TiRex-2` checkpoint. The Pro modules reuse the same `TimeseriesType` container used by the base forecast API.



## Installation

If you are running this notebook outside the development environment, install TiRex-2 with the example dependencies:

```bash
# from PyPI
pip install -q "tirex-2[examples]"

# or from this repository root
# %pip install -q -e ".[examples]"
```


In [ ]:

# Optional: set a Hugging Face token if your checkpoint is gated.
# os.environ["HF_TOKEN"] = "<insert-hf-token>"

import os
import tempfile

import numpy as np
import torch
import matplotlib.pyplot as plt

from tirex2 import TimeseriesType, load_model
from tirex2.demo import Demo
from tirex2.pro import (
    IncrementalForecaster,
    TimeSeriesAnomalyDetector,
    TimeSeriesRegressor,
)



## Load the pretrained TiRex-2 model

We load the public checkpoint once and reuse it for every Pro feature below.


In [ ]:

ckpt = "NX-AI/TiRex-2"
device = "cpu"  # use "cuda" if available

model = load_model(ckpt, device=device)
print("context length:", model.context_len)
print("max prediction length:", model.future_len)
print("quantiles:", [round(float(q), 3) for q in model.quantiles])



## 1. Streaming / incremental forecasting

`IncrementalForecaster` keeps a bounded rolling context buffer. When new observations arrive, it re-forecasts from the latest `context_length` steps instead of recomputing over the entire history.

This example seeds the forecaster with the last 96 steps of the holidays demo, produces a 24-step forecast, ingests 6 new observations, and forecasts again.


In [ ]:

demo = Demo.create_holidays_demo()

# Initial history can be longer than the rolling window; the wrapper keeps the last 96 steps.
history = TimeseriesType(
    target=torch.from_numpy(demo.target_context).unsqueeze(0),
    past_covariates=None,
    future_covariates=None,
)

forecaster = IncrementalForecaster(
    model,
    prediction_length=24,
    context_length=96,
    output_type="numpy",
)

forecaster.update(history)
forecast_0 = forecaster.forecast()
print("initial forecast shape:", forecast_0.shape, "(variates, quantiles, horizon)")

# Ingest the first 6 steps of the actual future and forecast again.
new_steps = 6
new_obs = TimeseriesType(
    target=torch.from_numpy(demo.target_future[:new_steps]).unsqueeze(0),
)
forecaster.update(new_obs)
forecast_1 = forecaster.forecast()
print("updated forecast shape:", forecast_1.shape)


In [ ]:

def plot_streaming_forecast(context, future, forecast, title):
    """Plot a rolling context, the actual future, and a quantile forecast."""
    fig, ax = plt.subplots(figsize=(10, 4))
    t_context = np.arange(len(context))
    t_future = np.arange(len(context), len(context) + len(future))
    t_fc = np.arange(len(context), len(context) + forecast.shape[-1])

    ax.plot(t_context, context, label="context", color="black")
    ax.plot(t_future, future, label="actual future", color="tab:green", alpha=0.8)

    median = forecast[0, 4, :]
    lower = forecast[0, 0, :]
    upper = forecast[0, -1, :]
    ax.plot(t_fc, median, label="forecast median", color="tab:blue")
    ax.fill_between(t_fc, lower, upper, alpha=0.3, color="tab:blue", label="10-90% band")

    ax.legend()
    ax.set_title(title)
    ax.set_xlabel("time step")
    plt.tight_layout()
    plt.show()

# Plot the first forecast against the full known future.
context_0 = history.target[0, -96:].numpy()
plot_streaming_forecast(context_0, demo.target_future, forecast_0, "Initial streaming forecast")

# The cached context after the update already includes the new observations.
cached_context = forecaster._cached.target[0].numpy()
remaining_future = demo.target_future[new_steps:]
plot_streaming_forecast(cached_context, remaining_future, forecast_1, "Forecast after ingesting new observations")



## 2. Time-series regression

`TimeSeriesRegressor` places a small MLP head on top of the frozen TiRex-2 backbone. It is useful when the downstream task is a scalar or vector target (e.g. remaining useful life, average demand over the next week) rather than a full quantile forecast.

Here we train the head to predict the mean value of the last 24 steps of a synthetic sine wave.


In [ ]:

def make_regression_dataset(n_samples=40, length=128, seed=42):
    rng = np.random.default_rng(seed)
    data = []
    for i in range(n_samples):
        t = np.linspace(0, 4 * np.pi * (i + 1), length)
        y = np.sin(t) + 0.1 * rng.normal(size=length)
        series = TimeseriesType(target=torch.from_numpy(y).astype(torch.float32).unsqueeze(0))
        # Scalar target per variate: mean of the last 24 steps.
        target = torch.tensor([[y[-24:].mean()]], dtype=torch.float32)
        data.append((series, target))
    return data

reg_data = make_regression_dataset()
train = reg_data[:32]
test = reg_data[32:]
print("train samples:", len(train), "test samples:", len(test))


In [ ]:

regressor = TimeSeriesRegressor(
    model,
    output_dim=1,
    hidden_dim=64,
    freeze_backbone=True,  # only the MLP head is trained
)

regressor.fit(
    train,
    epochs=5,
    batch_size=4,
    learning_rate=1e-3,
    context_length=96,
    prediction_length=1,
    output_dir="/tmp/tirex_regression_head",
    log_interval=5,
)


In [ ]:

# Evaluate on the held-out test set.
test_series = [ts for ts, _ in test]
test_targets = torch.stack([tgt for _, tgt in test])
predictions = torch.stack(regressor.predict(test_series))

mse = ((predictions - test_targets) ** 2).mean().item()
print(f"test MSE: {mse:.4f}")

# Demonstrate head save / reload.
regressor2 = TimeSeriesRegressor(model, output_dim=1, hidden_dim=64, freeze_backbone=True)
regressor2.load_head("/tmp/tirex_regression_head")
reloaded_predictions = torch.stack(regressor2.predict(test_series))
print("reloaded predictions match:", torch.allclose(predictions, reloaded_predictions, atol=1e-5))



## 3. Multivariate anomaly detection

`TimeSeriesAnomalyDetector` scores each time step by comparing the observed value to a one-step-ahead TiRex-2 quantile forecast. The threshold is calibrated on a reference series that is assumed to be mostly normal.

This example uses the `iqr_deviation` scorer and flags the top 1% of scores from the reference series. We then inject a spike and a level shift into a separate test series and check that the detector flags the anomalous region.


In [ ]:

def make_anomaly_series(length=250, inject=False, seed=7):
    rng = np.random.default_rng(seed)
    t = np.arange(length)
    y = 10.0 * np.sin(2 * np.pi * t / 50.0) + 0.5 * t / length + rng.normal(0, 0.3, length)
    if inject:
        y[180:183] += 6.0   # sharp spike
        y[220:] += 3.0      # level shift
    return torch.from_numpy(y.astype(np.float32)).unsqueeze(0)

clean = TimeseriesType(target=make_anomaly_series(inject=False))
test = TimeseriesType(target=make_anomaly_series(inject=True))

detector = TimeSeriesAnomalyDetector(
    model,
    prediction_length=1,
    scorer="iqr_deviation",
    aggregation="max",
    context_length=96,
)

threshold = detector.fit_threshold([clean], percentile=99.0)
print("calibrated threshold:", threshold)

result = detector.predict(test)
print("flagged time steps:", int(result.global_labels.sum().item()))


In [ ]:

fig, ax = plt.subplots(figsize=(12, 4))
series = test.target[0].numpy()
t = np.arange(len(series))

ax.plot(t, series, label="test series", color="black")
anomaly_idx = np.where(result.global_labels.numpy())[0]
if len(anomaly_idx):
    ax.scatter(anomaly_idx, series[anomaly_idx], color="red", zorder=5, label="flagged anomaly")

ax.axvspan(180, 183, color="orange", alpha=0.2, label="injected spike")
ax.axvline(220, color="green", linestyle="--", label="injected level shift")
ax.axhline(threshold, color="gray", linestyle=":", label="threshold")
ax.set_title("Anomaly detection on synthetic injected anomalies")
ax.set_xlabel("time step")
ax.legend()
plt.tight_layout()
plt.show()



## Summary

This notebook showed three TiRex-2 Pro workflows:

* **`IncrementalForecaster`** — bounded-context streaming updates without recomputing the full history.
* **`TimeSeriesRegressor`** — a frozen-backbone MLP head for scalar/vector targets, with save/load support.
* **`TimeSeriesAnomalyDetector`** — point-level anomaly scoring via one-step-ahead quantile forecasts and calibrated thresholds.

All three reuse the same pretrained TiRex-2 backbone and the same `TimeseriesType` inputs, so they compose naturally with the base forecast API.
